In [1]:
import os          
import pathlib     
                   
import sys         

here = pathlib.Path.cwd()      

ROOT = here.parents[2] if here.name == "day05" else here
os.chdir(ROOT)                 

SANDBOX = ROOT / "sandbox" / "w2" / "day05"

BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

리포지토리

- 연습 1

In [2]:
from datetime import date
from app.db.session import get_sessionmaker, get_engine
from app.models import Department, Document, DocumentVersion
from app.db.init_db import init_db

# 연습용 DB 경로
연습DB = SANDBOX / "repo_practice.db"
# 파일이 없어도 오류 발생하지 않도록 설정
연습DB.unlink(missing_ok=True)                          

# 연습용 DB 환경 설정
연습엔진 = get_engine(f"sqlite:///{연습DB}")
init_db(연습엔진)                   
연습세션 = get_sessionmaker(연습엔진)  

# SQL 연습
with 연습세션() as s:
    # 부서 저장
    s.add_all([
        Department(id="HRGA", name="인사총무"),
        Department(id="PU", name="구매팀"),
        Department(id="SE", name="보안팀"),
    ])
    # 쿼리문 나가서 DB에 저장
    s.flush()      
    # 문서 저장
    s.add_all([
        Document(id="DOC-HR-014", title="국내출장 여비 규정",
                 dept_id="HRGA", security_level="일반"),
        Document(id="DOC-PU-007", title="구매·계약 규정",
                 dept_id="PU", security_level="대외비"),
        Document(id="DOC-SE-003", title="정보보안 지침",
                 dept_id="SE", security_level="대외비"),
    ])
    # 쿼리문 나가서 DB에 저장
    s.flush()
    # 버전별 문서 저장
    s.add_all([
        DocumentVersion(doc_id="DOC-HR-014", version="v2.0", status="현행",
                        effective_from=date(2025, 7, 1), expires_at=None,
                        file_path="uploads/DOC-HR-014_v2.0.docx", file_format="docx"),
        DocumentVersion(doc_id="DOC-PU-007", version="v4.0", status="현행",
                        effective_from=date(2026, 3, 1), expires_at=None,
                        file_path="uploads/DOC-PU-007_v4.0.pdf", file_format="pdf"),
        DocumentVersion(doc_id="DOC-SE-003", version="v2.2", status="현행",
                        effective_from=date(2025, 10, 1), expires_at=None,
                        file_path="uploads/DOC-SE-003_v2.2.pdf", file_format="pdf"),
    ])
    # DB에 영구 저장
    s.commit()

print("연습 DB   :", 연습DB.relative_to(ROOT))
print("문서 3건 · 버전 3건 준비 완료 (대외비 2건 포함)")


연습 DB   : sandbox\w2\day05\repo_practice.db
문서 3건 · 버전 3건 준비 완료 (대외비 2건 포함)


In [3]:
from sqlalchemy import select

# 쿼리문 SQLAlchemy 2.x 버전으로 생성 
stmt = select(Document).where(Document.dept_id == "HRGA")

print(stmt)                                  
print()
print("바인딩된 값 :", stmt.compile().params)

SELECT documents.id, documents.title, documents.dept_id, documents.security_level, documents.owner_id, documents.created_at, documents.updated_at 
FROM documents 
WHERE documents.dept_id = :dept_id_1

바인딩된 값 : {'dept_id_1': 'HRGA'}


In [4]:
from app.repositories import document_repo

조건들 = [
    ("dept_id=SE", {"dept_id": "SE"}),
    ("security_level=대외비", {"security_level": "대외비"}),
    ("q=출장", {"q": "출장"}),
    ("dept_id=SE + status=현행", {"dept_id": "SE", "status": "출장"}),
]

with 연습세션() as session: 
    for 이름, 조건 in 조건들:
        # DB 조회한 결과 행들 변수에 담기
        결과행들 = document_repo.list_documents(session, **조건)
        뷰 = " . ".join(f"{v.doc_id} {v.version}" for v, d in 결과행들)
        print(f"{이름:} -> {len(결과행들)}건 {뷰}")


# DB 접속은 session
with 연습세션() as session:
    print("조건 없음")
    result_rows = document_repo.list_documents(session)
    # 몇개
    print(len(result_rows))
    for row in result_rows:
        print(row)
    vals = " , ".join(f"{v.doc_id} {v.version}" for v, d in result_rows)
    print(vals)

dept_id=SE -> 1건 DOC-SE-003 v2.2
security_level=대외비 -> 2건 DOC-PU-007 v4.0 . DOC-SE-003 v2.2
q=출장 -> 1건 DOC-HR-014 v2.0
dept_id=SE + status=현행 -> 0건 
조건 없음
3
(<app.models.document.DocumentVersion object at 0x000001F772C405C0>, <app.models.document.Document object at 0x000001F772C405F0>)
(<app.models.document.DocumentVersion object at 0x000001F772C40620>, <app.models.document.Document object at 0x000001F772C40680>)
(<app.models.document.DocumentVersion object at 0x000001F772C406B0>, <app.models.document.Document object at 0x000001F772C40710>)
DOC-HR-014 v2.0 , DOC-PU-007 v4.0 , DOC-SE-003 v2.2


In [5]:
from app.db.seed import count_rows, seed_all
from app.db.session import session_scope

# DB 초기화
init_db()

# 시드 데이터 저장
print("1회차 적재 = ", seed_all())
print("2회차 적재 = ", seed_all())
print("3회차 적재 = ", seed_all())

with session_scope() as session:
    print("직접 count : ", count_rows(session))

1회차 적재 =  {'departments': 6, 'users': 7, 'documents': 7, 'versions': 8}
2회차 적재 =  {'departments': 6, 'users': 7, 'documents': 7, 'versions': 8}
3회차 적재 =  {'departments': 6, 'users': 7, 'documents': 7, 'versions': 8}
직접 count :  {'departments': 6, 'users': 7, 'documents': 7, 'versions': 8}


- 서비스 사용

In [6]:
from app.services import document_service

목록 = document_service.list_documents(limit=20)
print("목록 개수 : ", len(목록), "개")
print("문서 1개 : ", 목록[0]["doc_id"], 목록[0]["version"], 목록[0]["title"])

문서 = document_service.get_document(doc_id="DOC-HR-014")
print("문서 : ", 문서["version"], 문서["status"])

목록 개수 :  8 개
문서 1개 :  DOC-HR-002 v3.1 복무 규정
문서 :  v2.0 현행
